# LingBot-Map: replicating upstream's *actual* demo configuration

Runs upstream's two demo scenes (`example/loop`, `example/courthouse`) under **both** inference
configurations described in the paper (arXiv 2604.14141), on a rented GPU, using this repo's
`recon/*.py` unmodified.

## Why there are two configs

The README's one-liner (`demo.py --image_folder example/courthouse --mask_sky`) is **not** the
pipeline that produced upstream's published demo videos. Those came from
`demo_render/batch_demo.py`, driven by `demo_render/process_videos.sh`, which is a different
program with different settings.

| | **A · Direct** (`demo.py`) | **B · VO** (`batch_demo.py`) |
| --- | --- | --- |
| paper section | §4.5 "Default Inference Configuration" — every benchmark number | §4.4 VO mode — *"for the large-scale demo videos … we use VO mode"* |
| mode | `streaming` | `windowed` |
| pose-reference window | k = 64 | k = 64 |
| keyframes | fixed, m = 1 | **adaptive optical flow**, 25.0 px, forced every 100 |
| window size | — | 64 keyframes |
| their input | `example/` folders | the source video, `TARGET_FRAMES=4000`, `IMAGE_STRIDE=1` |

The keyframe mechanism (paper §4.4) predicts pose and depth for each incoming frame, measures
optical flow against the most recent keyframe, and promotes the frame only once that flow clears
a threshold. **`demo.py` exposes it nowhere**, and `gct_stream.py` (Direct) does not implement it
at all — it lives only in `gct_stream_window.py`. So the README command is a strictly weaker
configuration than the one behind their clips, and every windowed run this project has logged so
far used fixed intervals instead.

`recon/reconstruct.py` now takes `--flow_threshold` / `--max_non_keyframe_gap` and passes them
through, so config B is reachable for the first time.

## The metric that decides it

Flow mode returns an `is_keyframe` mask, which the run record turns into **`keyframe_frac`**.

- `keyframe_frac` well below 1.0 → frames are dense enough that the selector is skipping some.
  The mechanism is doing its job.
- `keyframe_frac` **= 1.0** → every frame cleared a 25 px flow threshold, so consecutive *inputs*
  are already further apart than upstream's *keyframe* spacing, and there is no densely-tracked
  frame anywhere in between. That is a property of the footage that no config can undo.

Measured on the shipped frames (phase correlation, scaled to the real 518 px width): courthouse
consecutive frames sit **~47 px** apart, loop **~2 px**. So the expectation going in is
`keyframe_frac ≈ 1.0` for courthouse and clearly below it for loop. Stated up front so the run
can contradict it.

## What else it does

A `kv_cache_sliding_window` ladder (16 → 128) under config A, heavy Open3D cleanup of every run,
renders of each cleaned cloud, and pasteable `notes/experiments.md` rows.

> **Runtime → Change runtime type → A100 or L4 first.** On a T4 (Turing) `reconstruct.py` drops
> to fp16 instead of bf16, changing the numeric path as well as the VRAM.

## 1 · Which GPU did we get?

In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU (A100 or L4)."
P = torch.cuda.get_device_properties(0)
VRAM_GB = P.total_memory / 1e9
CAP = torch.cuda.get_device_capability()
print(f"\n{P.name}  {VRAM_GB:.1f} GB  sm_{CAP[0]}{CAP[1]}  torch {torch.__version__}")
print(f"local box for comparison: RTX 4060 Ti, 8.6 GB, sm_89  ->  {VRAM_GB/8.6:.1f}x the VRAM")

# reconstruct.py picks bf16 on sm_80+ and fp16 below it. The paper specifies bfloat16, so a
# pre-Ampere card is not a replication -- it changes precision at the same time as VRAM.
if CAP[0] < 8:
    print("\nWARNING: pre-Ampere -> fp16, but the paper specifies bfloat16. Not a clean replication.")
if VRAM_GB < 20:
    print("WARNING: <20 GB. kvsw 128 will probably OOM; everything else should fit.")

## 2 · Configuration

Both configs below are transcribed from the paper and from `demo_render/process_videos.sh`. The
only departures are on the **export** side — how points are selected out of the finished
predictions — and they cannot affect geometry or poses:

- `--pixel_stride 2` is a spatial subsample of the exported cloud (upstream's renderer voxelises
  at 1 mm and our cleanup voxelises at 2 cm, so this changes nothing downstream).
- confidence: ours is a percentile, upstream's is absolute (`1.5` in `demo.py`'s viewer, `2.0` in
  their renderer). Set `CONF_ABS = 2.0` to match theirs exactly; left on the percentile by default
  so these runs stay directly comparable to the ones already in `notes/experiments.md`.

`keyframe_interval` is pinned to 1 rather than left on auto. Upstream's auto is
`(n + 319) // 320`; ours is `ceil(n / 240)`. They agree on loop's 237 frames and **disagree** on
courthouse's 286, so auto would have silently changed the thing being measured.

In [ ]:
SCENES     = ["loop", "courthouse"]   # upstream also ships "university" (324 frames)
CHECKPOINT = "lingbot-map.pt"         # paper/benchmark/demo checkpoint

# ── A · Direct: paper sec 4.5, "Default Inference Configuration" ─────────────
# "Direct Output Mode with a local pose-reference window size k=64 and keyframe
#  interval m=1, at a resolution of 518x518 with bfloat16 precision."
PAPER_DIRECT = dict(
    mode="streaming",
    kv_cache_sliding_window=64,      # k
    keyframe_interval=1,             # m
    num_scale_frames=8,
    camera_num_iterations=4,
    image_size=518,
    preprocess_mode="crop",          # demo.load_images hardcodes crop
)

# ── B · VO: paper sec 4.4, parameterised by demo_render/process_videos.sh ────
#   MODE="windowed"  WINDOW_SIZE=64  FLOW_THRESHOLD=25.0
#   MAX_NON_KEYFRAME_GAP=100  IMAGE_STRIDE=1
# overlap_keyframes=-1 means "unset", which is what batch_demo.py passes; the model
# then resolves it to num_scale_frames internally.
PAPER_VO = dict(
    mode="windowed",
    window_size=64,
    overlap_keyframes=-1,
    kv_cache_sliding_window=64,
    num_scale_frames=8,
    keyframe_interval=1,             # ignored once flow_threshold > 0
    flow_threshold=25.0,
    max_non_keyframe_gap=100,
    camera_num_iterations=4,
    image_size=518,
    preprocess_mode="crop",
)

MASK_SKY = {"courthouse": True, "university": True, "loop": False}   # per upstream's README

# ── export (post-inference; cannot affect poses or drift) ───────────────────
CONF_PERCENTILE = 55
CONF_ABS        = None   # set 2.0 for upstream's renderer vis_threshold
PIXEL_STRIDE    = 2
VRAM_FRACTION   = 0.92

# ── cache ladder, config A ─────────────────────────────────────────────────
RUN_SWEEP   = True
SWEEP_SCENE = "courthouse"
SWEEP       = [(16, 1), (16, 2), (24, 1), (32, 1), (64, 1), (128, 1)]

# ── cleanup / output ───────────────────────────────────────────────────────
CLEAN_HEAVY     = True       # std-ratio 2.0->1.5, min-neighbors 12->16
CAMERA_HEIGHT_M = 1.5
SAVE_TO_DRIVE   = False
KEEP_RAW_PLY    = False

VIDEOS    = {}               # {"my_trail": "/content/drive/MyDrive/trail.MOV"}
VIDEO_FPS = 5

WORK = "/content/gd"
print("config loaded")

## 3 · Install

`lingbot-map`'s `pyproject.toml` does not pin torch, so this installs on top of whatever torch
Colab ships and leaves the CUDA stack alone. If pip asks you to restart, do it and re-run from
cell 1 — the clone and downloads are already on disk and get skipped.

FlashInfer is deliberately not installed: it is upstream's attention *kernel*, not a different
attention, and `reconstruct.py` passes `use_sdpa=True` so PyTorch's SDPA computes the same thing.
Every run prints `flashinfer not available`; that line is expected.

In [ ]:
import os, pathlib, subprocess, sys

LINGBOT_SRC = "/content/lingbot-map"
os.environ["LINGBOT_SRC"] = LINGBOT_SRC   # reconstruct.py reads this to import upstream demo.py

if not pathlib.Path(LINGBOT_SRC, ".git").exists():
    !git clone --depth 1 https://github.com/Robbyant/lingbot-map.git {LINGBOT_SRC}
else:
    print("lingbot-map already cloned")

# `!pip` not `%pip`: the line magic does not expand {LINGBOT_SRC}.
!pip install -q -e "{LINGBOT_SRC}[vis]"
!pip install -q open3d imageio-ffmpeg

import open3d as o3d
print("open3d", o3d.__version__)
print("frames on disk:", {d.name: len(list(d.glob('*.png')))
                          for d in sorted(pathlib.Path(LINGBOT_SRC, "example").iterdir()) if d.is_dir()})

## 4 · Get the GeologicDome `recon/` scripts

**Use a fresh `recon.zip`** — `--flow_threshold` / `--max_non_keyframe_gap` / `--conf_threshold`
were added for this notebook, and an older zip silently runs config A twice.

```powershell
Compress-Archive -Path recon\*.py -DestinationPath recon.zip -Force
```

The repo is private, so the cell tries Drive, then a `GD_TOKEN` Colab Secret, then upload.

In [ ]:
import pathlib, shutil, zipfile

RECON = pathlib.Path("/content/recon")
NEEDED = ["reconstruct.py", "calibrate_scale.py", "clean_cloud.py", "inspect_cloud.py",
          "extract_frames.py"]


def _ok(d):
    return d.is_dir() and all((d / n).exists() for n in NEEDED)


if not _ok(RECON):
    for c in ["/content/drive/MyDrive/GeologicDome/recon", "/content/drive/MyDrive/recon"]:
        if _ok(pathlib.Path(c)):
            shutil.copytree(c, RECON, dirs_exist_ok=True)
            print("copied from Drive:", c)
            break

if not _ok(RECON):
    try:
        from google.colab import userdata
        tok = userdata.get("GD_TOKEN")
        url = f"https://{tok}@github.com/adikothuri3/geologic_dome_sim_onboarding.git"
        subprocess.run(["git", "clone", "--depth", "1", url, "/content/gd_repo"], check=True)
        shutil.copytree("/content/gd_repo/recon", RECON, dirs_exist_ok=True)
        print("cloned private repo")
    except Exception as e:
        print("no token clone:", type(e).__name__)

if not _ok(RECON):
    from google.colab import files
    print("Upload recon.zip  (PowerShell: Compress-Archive -Path recon\\*.py -DestinationPath recon.zip -Force)")
    up = files.upload()
    name = next(iter(up))
    with zipfile.ZipFile(name) as z:
        z.extractall("/content/_up")
    src = next(p.parent for p in pathlib.Path("/content/_up").rglob("reconstruct.py"))
    shutil.copytree(src, RECON, dirs_exist_ok=True)

assert _ok(RECON), f"missing {[n for n in NEEDED if not (RECON / n).exists()]}"

# Hard gate: config B is unreachable without these, and failing here beats discovering
# it three runs later when every 'VO' result is silently a Direct-mode duplicate.
helptext = subprocess.run([sys.executable, str(RECON / "reconstruct.py"), "--help"],
                          capture_output=True, text=True,
                          env={**os.environ, "LINGBOT_SRC": LINGBOT_SRC}).stdout
for flag in ("--flow_threshold", "--max_non_keyframe_gap", "--conf_threshold"):
    assert flag in helptext, f"{flag} missing -- this is an OLD recon.zip, re-zip from the repo"
print("pipeline code ready, flow-keyframe flags present")

## 5 · Checkpoint + sky segmentation

4.63 GB from HuggingFace. `skyseg.onnx` is needed for courthouse (upstream runs that scene with
`--mask_sky`, and their demo pipeline masks sky on every video); it runs on CPU by design.

In [ ]:
import pathlib
from huggingface_hub import hf_hub_download

CKPT_DIR = pathlib.Path("/content/ckpt"); CKPT_DIR.mkdir(exist_ok=True)
CKPT = pathlib.Path(hf_hub_download("robbyant/lingbot-map", CHECKPOINT, local_dir=str(CKPT_DIR)))
print(f"{CKPT}  {CKPT.stat().st_size/1e9:.2f} GB")

SKYSEG = CKPT_DIR / "skyseg.onnx"
if any(MASK_SKY.get(s) for s in SCENES) and not SKYSEG.exists():
    !curl -sL -o {SKYSEG} https://huggingface.co/JianyuanWang/skyseg/resolve/main/skyseg.onnx
    print(f"skyseg.onnx  {SKYSEG.stat().st_size/1e6:.0f} MB")

## 6 · The runner

Shells out to `recon/reconstruct.py` so the code path is identical to a local run. Output is teed
to a log — a truncated notebook cell is not evidence, and piping this into anything that swallows
the exit code is how an OOM-killed run gets mistaken for a success.

In [ ]:
import json, pathlib, subprocess, sys, time

WORKP = pathlib.Path(WORK)
(WORKP / "runs").mkdir(parents=True, exist_ok=True)
(WORKP / "logs").mkdir(parents=True, exist_ok=True)
RUNS = {}     # tag -> run.json


def frames_dir(scene):
    d = pathlib.Path(LINGBOT_SRC, "example", scene)
    if d.is_dir():
        return d
    d = WORKP / "frames" / scene
    assert d.is_dir(), f"no frames for {scene!r}"
    return d


def reconstruct(scene, tag, base, quiet=True, **over):
    """Run one reconstruction. `base` is PAPER_DIRECT or PAPER_VO; `over` overrides it."""
    cfg = dict(base); cfg.update(over)
    out = WORKP / "runs" / tag
    log = WORKP / "logs" / f"{tag}.log"

    if (out / "run.json").exists():
        rec = json.loads((out / "run.json").read_text())
        print(f"[{tag}] already done, reusing")
        RUNS[tag] = rec
        return rec

    cmd = [sys.executable, str(RECON / "reconstruct.py"),
           "--frames", str(frames_dir(scene)), "--out", str(out),
           "--model_path", str(CKPT),
           "--pixel_stride", str(PIXEL_STRIDE),
           "--vram_fraction", str(VRAM_FRACTION)]
    cmd += (["--conf_threshold", str(CONF_ABS)] if CONF_ABS is not None
            else ["--conf_percentile", str(CONF_PERCENTILE)])
    for k, v in cfg.items():
        if v is not None:
            cmd += [f"--{k}", str(v)]
    if MASK_SKY.get(scene) and SKYSEG.exists():
        cmd += ["--mask_sky", "--skyseg_model", str(SKYSEG)]

    flow = cfg.get("flow_threshold", 0)
    print(f"[{tag}] {cfg['mode']}  kvsw={cfg['kv_cache_sliding_window']}  "
          + (f"flow={flow}px/gap={cfg['max_non_keyframe_gap']}" if flow
             else f"kfi={cfg['keyframe_interval']}")
          + ("  +sky" if MASK_SKY.get(scene) else ""))
    t0 = time.time()
    with open(log, "w") as fh:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             bufsize=1, env={**os.environ, "LINGBOT_SRC": LINGBOT_SRC})
        for line in p.stdout:
            fh.write(line)
            if not quiet:
                sys.stdout.write(line)
        rc = p.wait()

    if rc != 0:
        print(f"[{tag}] FAILED rc={rc} -- tail of {log}:")
        print("".join(log.read_text().splitlines(keepends=True)[-15:]))
        return None

    rec = json.loads((out / "run.json").read_text())
    RUNS[tag] = rec
    kf = rec.get("keyframe_frac")
    print(f"[{tag}] {time.time()-t0:.0f}s  {rec['n_points']:,} pts  "
          f"{rec['peak_vram_gb']:.2f} GB  ratio {rec['traj_length_over_extent']}"
          + (f"  keyframes {rec['n_keyframes']}/{rec['n_frames']} ({kf:.0%})" if kf else ""))
    return rec


for name, path in VIDEOS.items():
    d = WORKP / "frames" / name
    if not d.is_dir():
        subprocess.run([sys.executable, str(RECON / "extract_frames.py"), path, str(d),
                        "--fps", str(VIDEO_FPS)], check=True)
    if name not in SCENES:
        SCENES.append(name)

print("runner ready")

## 7 · Config A — Direct mode (the paper's benchmark configuration)

`traj_length_over_extent` is camera-path length over scene size. It is a drift **detector**, not a
quality score — a 2026-08-05 run scored 2.87 while looking terrible — but a value in the twenties
means the poses have collapsed.

Reference: locally, loop = **3.36** at kvsw 24, courthouse = **24.88** at kvsw 16.

In [ ]:
LOCAL = {"loop": dict(ratio=3.36, kvsw=24), "courthouse": dict(ratio=24.88, kvsw=16)}

for scene in SCENES:
    reconstruct(scene, f"{scene}_direct", PAPER_DIRECT)

print()
for scene in SCENES:
    r = RUNS.get(f"{scene}_direct")
    if r is None:
        print(f"{scene:12s} FAILED"); continue
    base = LOCAL.get(scene, {}).get("ratio")
    note = f"   local {base} at kvsw {LOCAL[scene]['kvsw']}" if base else ""
    print(f"{scene:12s} ratio {r['traj_length_over_extent']:6.2f}   {r['n_points']:>10,} pts   "
          f"{r['peak_vram_gb']:5.2f} GB   {r['fps']:.2f} fps{note}")

## 8 · Config B — VO mode with adaptive flow keyframes

This is upstream's demo-video configuration, reachable for the first time. Read
**`keyframe_frac`** before anything else: it is the direct test of whether these frames are dense
enough for the mechanism to have anything to select from.

`n_windows_stitched` and `window_scale_span` matter too — VO fuses windows by Sim(3) alignment
over their overlap, and the paper is explicit that this *adds* drift at each boundary
(§4.4: *"VO mode incurs extra alignment error that compounds with the number of windows"*). A
scale span far from 1.0 means that alignment failed.

In [ ]:
for scene in SCENES:
    reconstruct(scene, f"{scene}_vo", PAPER_VO)

print()
print(f"{'scene':12} {'ratio':>7} {'keyframes':>16} {'windows':>8} {'scale span':>11} {'peak GB':>8}")
for scene in SCENES:
    r = RUNS.get(f"{scene}_vo")
    if r is None:
        print(f"{scene:12} FAILED"); continue
    kf = r.get("keyframe_frac")
    kf_txt = f"{r['n_keyframes']}/{r['n_frames']} ({kf:.0%})" if kf is not None else "n/a"
    print(f"{scene:12} {r['traj_length_over_extent']:>7.2f} {kf_txt:>16} "
          f"{r['n_windows_stitched']:>8} {r['window_scale_span']:>11.2f} {r['peak_vram_gb']:>8.2f}")

print("\nA vs B, same frames, same card:")
for scene in SCENES:
    a, b = RUNS.get(f"{scene}_direct"), RUNS.get(f"{scene}_vo")
    if a and b:
        print(f"  {scene:12} Direct {a['traj_length_over_extent']:6.2f}  ->  "
              f"VO {b['traj_length_over_extent']:6.2f}")

## 9 · Cache ladder (config A)

Locally this is unanswerable — 24 is the ceiling, so the ladder has one rung.

The pair to read is **`kvsw 16 / kfi 2`** against **`kvsw 32 / kfi 1`**: same span of footage,
half the cached views. If only the view count matters, `(32, 1)` wins and VRAM is the whole lever.
If they land together, the model wants horizon, and buying VRAM will not fix long walks.

In [ ]:
sweep_rows = []
if RUN_SWEEP:
    for kvsw, kfi in SWEEP:
        if kvsw > 64 and VRAM_GB < 20:
            print(f"skipping kvsw {kvsw} on a {VRAM_GB:.0f} GB card"); continue
        if (kvsw, kfi) == (PAPER_DIRECT["kv_cache_sliding_window"],
                           PAPER_DIRECT["keyframe_interval"]):
            r = RUNS.get(f"{SWEEP_SCENE}_direct")        # already paid for in cell 7
        else:
            r = reconstruct(SWEEP_SCENE, f"{SWEEP_SCENE}_kv{kvsw}_kfi{kfi}", PAPER_DIRECT,
                            kv_cache_sliding_window=kvsw, keyframe_interval=kfi)
        sweep_rows.append((kvsw, kfi, r))

    print(f"\n{SWEEP_SCENE}: cache sweep")
    print(f"{'kvsw':>5} {'kfi':>4} {'ratio':>8} {'peak GB':>8} {'fps':>6} {'points':>11}")
    for kvsw, kfi, r in sweep_rows:
        if r is None:
            print(f"{kvsw:>5} {kfi:>4} {'OOM/FAIL':>8}"); continue
        print(f"{kvsw:>5} {kfi:>4} {r['traj_length_over_extent']:>8.2f} {r['peak_vram_gb']:>8.2f} "
              f"{r['fps']:>6.2f} {r['n_points']:>11,}")

## 10 · Verdict

Decided from numbers, not from how the renders feel. `traj_length_over_extent` ≤ 6 is the bar
`calibrate_scale.py` requires before it will trust poses enough to anchor metric scale.

In [ ]:
print("=" * 74)
for scene in SCENES:
    a, b = RUNS.get(f"{scene}_direct"), RUNS.get(f"{scene}_vo")
    if not (a and b):
        continue
    best = min(a["traj_length_over_extent"], b["traj_length_over_extent"])
    kf = b.get("keyframe_frac")
    print(f"\n{scene}:  Direct {a['traj_length_over_extent']:.2f}   "
          f"VO+flow {b['traj_length_over_extent']:.2f}   best {best:.2f}")
    if kf is not None:
        if kf >= 0.99:
            print(f"  keyframe_frac {kf:.0%} -- EVERY frame cleared the 25 px flow threshold.")
            print("  These frames are sampled more sparsely than upstream's own keyframe")
            print("  spacing, so there is no densely-tracked frame in between and the")
            print("  selector has nothing to select. This is an input property, not a config.")
        else:
            print(f"  keyframe_frac {kf:.0%} -- the selector is genuinely skipping frames,")
            print("  so the footage is inside the regime the mechanism was built for.")
    print("  => " + ("USABLE (ratio <= 6, scale can be anchored)" if best <= 6
                     else "COLLAPSED (ratio > 6; calibrate_scale.py will refuse this run)"))

r16 = next((r["traj_length_over_extent"] for k, f, r in sweep_rows if (k, f) == (16, 1) and r), None)
ch = RUNS.get("courthouse_direct")
if ch and r16:
    print(f"\ncache ladder control: kvsw 16 here = {r16:.2f}, on the local 8 GB box = 24.88")
    print(f"                     kvsw 64 here = {ch['traj_length_over_extent']:.2f}")
    print("  If those two agree, the card was never the variable.")
print("=" * 74)

## 11 · Heavy Open3D cleanup

Two scripts, unmodified:

- **`calibrate_scale.py`** — fits candidate ground planes, keeps the one holding camera height
  *constant* (inlier count picks a wall in a corridor), divides median camera height by an assumed
  1.5 m eye height. It **refuses** above drift ratio 6, because drifted poses cannot anchor
  anything. Whether courthouse now clears that gate is itself a result.
- **`clean_cloud.py --scale auto`** — scales to metres first so every filter size is a real
  distance, then statistical outlier removal → radius filtering (the flying-pixel streaks shed at
  occlusion edges, which statistical removal misses because each streak is locally dense along its
  own filament) → 2 cm voxel downsample → ground plane to +Z at z=0.

In [ ]:
import json, subprocess, sys

MAIN_TAGS = [f"{s}_{k}" for s in SCENES for k in ("direct", "vo")]


def clean(tag):
    d = WORKP / "runs" / tag
    if not (d / "cloud.ply").exists():
        return None
    if (d / "clean_stats.json").exists():
        print(f"[{tag}] already cleaned"); return json.loads((d / "clean_stats.json").read_text())

    cal = subprocess.run([sys.executable, str(RECON / "calibrate_scale.py"), str(d),
                          "--camera-height", str(CAMERA_HEIGHT_M)], capture_output=True, text=True)
    print(f"── {tag}: scale")
    print(cal.stdout.strip())
    if cal.returncode != 0:
        tail = cal.stderr.strip().splitlines()
        print("  REFUSED:", tail[-1] if tail else f"rc={cal.returncode}")

    cmd = [sys.executable, str(RECON / "clean_cloud.py"), str(d)]
    if (d / "scale.json").exists():
        cmd += ["--scale", "auto"]
    else:
        print("  no scale.json -> cleaning in arbitrary units (NOT terrain-ready)")
    if CLEAN_HEAVY:
        cmd += ["--std-ratio", "1.5", "--min-neighbors", "16"]

    cl = subprocess.run(cmd, capture_output=True, text=True)
    print(f"── {tag}: clean"); print(cl.stdout.strip() or cl.stderr.strip())
    return json.loads((d / "clean_stats.json").read_text()) if cl.returncode == 0 else None


cleaned = {t: clean(t) for t in MAIN_TAGS if t in RUNS}

## 12 · Look at all four maps

Direct and VO side by side for each scene. Tries `inspect_cloud.py` first (four orbit views
through Open3D's headless EGL path, plus `inspect_stats.json`); Colab does not always expose EGL
to Open3D, so there is a matplotlib fallback that plots the same four views.

In [ ]:
import numpy as np, subprocess, sys
import matplotlib.pyplot as plt
from IPython.display import Image, display


def mpl_preview(ply, title, max_pts=250_000):
    pcd = o3d.io.read_point_cloud(str(ply))
    p, c = np.asarray(pcd.points), np.asarray(pcd.colors)
    if len(p) > max_pts:
        i = np.random.default_rng(0).choice(len(p), max_pts, replace=False)
        p, c = p[i], (c[i] if len(c) else c)
    c = c if len(c) else np.full((len(p), 3), 0.35)

    th = np.deg2rad(45)
    rot = p @ np.array([[np.cos(th), -np.sin(th), 0], [np.sin(th), np.cos(th), 0], [0, 0, 1]]).T
    views = [("top  (x,y)", p[:, 0], p[:, 1]), ("front (x,z)", p[:, 0], p[:, 2]),
             ("side  (y,z)", p[:, 1], p[:, 2]), ("oblique", rot[:, 0], rot[:, 2])]

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    for ax, (name, u, v) in zip(axes.ravel(), views):
        ax.scatter(u, v, c=np.clip(c, 0, 1), s=0.06, linewidths=0)
        ax.set_aspect("equal"); ax.set_title(name, fontsize=10); ax.tick_params(labelsize=7)
    fig.suptitle(title, fontsize=13); fig.tight_layout(); plt.show()


for tag in MAIN_TAGS:
    d = WORKP / "runs" / tag
    ply = d / "cloud_clean.ply"
    if not ply.exists():
        continue
    st = json.loads((d / "clean_stats.json").read_text())
    r = RUNS[tag]
    print(f"\n{'='*74}\n{tag}:  ratio {r['traj_length_over_extent']}   "
          f"{st['n_out']:,} points   extent {st['extent_out']} {st['units']}"
          + (f"   keyframes {r['keyframe_frac']:.0%}" if r.get("keyframe_frac") else "")
          + f"\n{'='*74}")

    shown = False
    try:
        res = subprocess.run([sys.executable, str(RECON / "inspect_cloud.py"), str(d),
                              "--clean", "--tag", "clean_"], capture_output=True, text=True,
                             timeout=1200)
        pngs = sorted(d.glob("view_clean_*.png"))
        if res.returncode == 0 and pngs:
            for q in pngs:
                display(Image(filename=str(q), width=760))
            shown = True
    except Exception as e:
        print("inspect_cloud unavailable:", type(e).__name__)
    if not shown:
        print("(Open3D offscreen EGL unavailable -- matplotlib fallback)")
        mpl_preview(ply, tag)

## 13 · Take the results home

Zips per-run artifacts and prints `notes/experiments.md` rows. Rows are mandatory per `CLAUDE.md`
— every reconstruction run gets one, failures included. Fill in `commit` with the short hash of
the `recon/` you zipped.

In [ ]:
import shutil, zipfile, datetime

STAMP = datetime.date.today().isoformat()
bundle = pathlib.Path(f"/content/lingbotmap_colab_{STAMP}.zip")

KEEP = ["run.json", "scale.json", "clean_stats.json", "inspect_stats.json",
        "cloud_clean.ply", "trajectory.npz"]
if KEEP_RAW_PLY:
    KEEP.append("cloud.ply")

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for d in sorted((WORKP / "runs").iterdir()):
        for n in KEEP:
            if (d / n).exists():
                z.write(d / n, f"{d.name}/{n}")
        for q in d.glob("view_*.png"):
            z.write(q, f"{d.name}/{q.name}")
    for q in (WORKP / "logs").glob("*.log"):
        z.write(q, f"logs/{q.name}")

print(f"{bundle}  {bundle.stat().st_size/1e6:.1f} MB")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = pathlib.Path("/content/drive/MyDrive/GeologicDome/colab_runs")
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(bundle, dst); print("copied to", dst)

gpu = P.name.replace("NVIDIA ", "")
print("\n" + "=" * 74 + "\npaste into notes/experiments.md:\n")
for tag in sorted(RUNS):
    r = RUNS[tag]
    scene = tag.split("_")[0]
    if r.get("flow_threshold"):
        how = (f"**VO/windowed ws={r['window_size']}**, flow {r['flow_threshold']:g} px / gap "
               f"{r['max_non_keyframe_gap']}, keyframes {r['n_keyframes']}/{r['n_frames']} "
               f"({r['keyframe_frac']:.0%})")
    else:
        how = f"Direct/streaming, kfi={r['keyframe_interval']}"
    cfg = (f"LingBot-Map base on **Colab {gpu} {VRAM_GB:.0f} GB**, upstream `example/{scene}` "
           f"({r['n_frames']} frames), {how}, kvsw={r['kv_cache_sliding_window']}, "
           f"nsf={r['num_scale_frames']}, 518 crop")
    met = (f"{r['inference_s']:.0f} s, {r['fps']:.2f} fps, peak VRAM {r['peak_vram_gb']:.2f} GB, "
           f"{r['n_points']:,} pts, **ratio {r['traj_length_over_extent']}**")
    print(f"| {STAMP}-colab-{tag.replace('_','-')} | <hash> | {cfg} | — | {met} | <takeaway> |")

from google.colab import files
files.download(str(bundle))